# Flat 4-in-a-Row — Colab GPU trainingTrains `run_ftf_005` (first-to-four mode) on a Colab GPU, seeded from the bestsnapshot of `run_ftf_004`.**Run the cells top to bottom.** Cells 1–6 are setup and take about two minutes.Cell 8 launches training in the background, so you can close the tab and comeback to cell 9 to check on it.### How persistence worksTraining writes to ordinary folders inside the cloned repo, and a backgroundloop mirrors them to Google Drive every two minutes. If the runtime is recycledyou lose at most two minutes of progress; re-running cells 1–8 in a new sessionrestores from Drive and picks up from the last snapshot automatically.### Session budgetA snapshot is written every 1000 games. Free Colab caps sessions at roughly 12hours and disconnects after ~90 minutes idle, so expect to need several sessionsto finish 20,000 games. That is fine and fully supported — cell 8 detects anexisting run and resumes it.---> ### Prerequisite: push the code first>> Cell 4 clones this project from GitHub, so **the `submission` branch must> already contain the training changes this notebook depends on** — the> `FLAT4_DEVICE` device override, the CPU pinning for episode workers, and the> `--benchmark-every` flag.>> If those are still sitting uncommitted on your laptop, cell 8 will fail> immediately with `unrecognized arguments: --benchmark-every`.>> Verify from the laptop before starting a session:>> ```> git log origin/submission -1 --stat -- src/training/train.py> ```

## 1. Runtime checkConfirms a GPU is attached and reports the CPU/RAM you actually got.

In [ ]:
import multiprocessing, shutil, subprocess

print("=== GPU ===")
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True,
)
if gpu.returncode == 0 and gpu.stdout.strip():
    print(gpu.stdout.strip())
else:
    print("NO GPU DETECTED.")
    print("Runtime > Change runtime type > Hardware accelerator: T4 GPU, then re-run.")

print("\n=== CPU / RAM ===")
print("vCPUs:", multiprocessing.cpu_count())
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout.strip())

total, used, free = shutil.disk_usage("/content")
print(f"\n=== Disk (/content) ===\nfree {free / 2**30:.1f} GiB of {total / 2**30:.1f} GiB")

try:
    import torch
    print(f"\ntorch {torch.__version__}  cuda_available={torch.cuda.is_available()}")
except ImportError:
    print("\ntorch not installed yet (cell 4 handles it)")

## 2. Run configurationThe only cell you would normally edit.`SEED_WEIGHTS` points at `run_ftf_004/gen_009` rather than that run's`best/weights.pt`. This is deliberate: `run_ftf_004` was executed in twosittings, and the second sitting reset its internal best-score tracker, so itoverwrote `best/weights.pt` at game 16,300 with a model whose mean episodelength was ~19. The genuine best was around game 9,000 at ~8.9, which is what`gen_009` holds.

In [ ]:
REPO_URL   = "https://github.com/Colile1/ITRI616_AI_Game_Project_Flat_4_in_Row.git"
BRANCH     = "submission"
WORK_DIR   = "/content/flat4"
DRIVE_DIR  = "/content/drive/MyDrive/ITRI616_flat4"

RUN_ID     = "run_ftf_005"
MODE       = "first_to_four"
BOARD_SIZE = 8

# Seed from run_ftf_004/gen_009 — 9,000 games, mean_ep_len 8.93, WR_h 17%, Elo 782.7
SEED_WEIGHTS = "models/size_08/run_ftf_004/gen_009/weights.pt"
SEED_GAME    = 9000
SEED_ELO     = 782.7

TOTAL_GAMES  = 20000     # absolute target, not "games from here"
LR_START     = 2e-4      # fine-tuning LR — full 1e-3 would wipe the seeded policy
SEED         = 1
BENCHMARK    = "alphabeta_d4"

# Games between benchmark checks. The project default is 100, but one check
# plays BENCHMARK_GAMES_PER_CHECK=128 games against a depth-4 alpha-beta search
# — measured at well over 15 minutes on a local CPU, and Colab's cores are
# slower. At the default this single diagnostic would eat a large share of the
# session. It has no effect on what the agent learns, so 500 trades benchmark
# curve resolution for real training throughput.
#
# For context on where the time actually goes: every 100 training games the
# loop also plays 340 evaluation games (200 vs random + 100 vs heuristic + 40
# for Elo). Those feed the metrics the report is built on, so they are left at
# their configured values — but it means wall-clock time is dominated by
# measurement, not by learning.
BENCHMARK_EVERY = 500

# Leave at 1 unless you have read this note.
#
# --workers > 1 does more than parallelise: the parallel code path in train.py
# forces every opponent to HeuristicAgent, bypassing self-play and the snapshot
# pool entirely. That is a different training curriculum, not a faster version
# of the same one, and results would not be comparable to run_ftf_004.
# Free-tier Colab also only gives 2 vCPUs, so the speedup is small anyway.
WORKERS      = 1

SYNC_SECONDS = 120       # how often the background job mirrors to Drive

print(f"{RUN_ID}: seed from game {SEED_GAME} -> target {TOTAL_GAMES} games")

## 3. Mount Google DriveAuthorise when prompted. Everything is written under a single `ITRI616_flat4` folder.

In [ ]:
import os

from google.colab import drive
drive.mount("/content/drive")

for sub in ("", "/logs", f"/models/size_{BOARD_SIZE:02d}", f"/results/size_{BOARD_SIZE:02d}"):
    os.makedirs(DRIVE_DIR + sub, exist_ok=True)

print("Drive ready at", DRIVE_DIR)

## 4. Clone the repository and install dependenciesColab already ships torch (CUDA build), numpy and matplotlib, so this onlyinstalls what is genuinely missing. `pygame` is deliberately skipped — thetraining code path never imports it, only the interactive UI does.

In [ ]:
import subprocess, sys, os

if os.path.isdir(f"{WORK_DIR}/.git"):
    print("Repo already present — fetching latest.")
    subprocess.run(["git", "fetch", "origin"], cwd=WORK_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=WORK_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=WORK_DIR, check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)   # so `import src...` resolves

missing = []
for mod, spec in (("numpy", "numpy>=1.26"), ("matplotlib", "matplotlib>=3.8"), ("torch", "torch>=2.2")):
    try:
        __import__(mod)
    except ImportError:
        missing.append(spec)

if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All training dependencies already present.")

import torch
print(f"\ncwd: {os.getcwd()}")
print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
print("seed weights present:", os.path.isfile(SEED_WEIGHTS))

# Fail fast on a stale clone rather than letting cell 8 die with an argparse
# error 20 seconds after it detaches into the background.
required = {
    "--benchmark-every flag (src/training/train.py)":
        ("src/training/train.py", "--benchmark-every"),
    "FLAT4_DEVICE override (src/agents/dqn_agent.py)":
        ("src/agents/dqn_agent.py", "FLAT4_DEVICE"),
}
absent = [label for label, (path, needle) in required.items()
          if needle not in open(os.path.join(WORK_DIR, path)).read()]
if absent:
    raise SystemExit(
        "The cloned branch predates changes this notebook needs:\n  - "
        + "\n  - ".join(absent)
        + f"\n\nPush those commits to origin/{BRANCH} from your laptop, "
          "then re-run this cell."
    )
print("Required training features present in the clone.")

## 5. Restore previous progress from DriveOn a first run this does nothing. On a later session it copies the run's modelsnapshots and result logs back out of Drive so that `--resume` can find them.

In [ ]:
import subprocess, os

def _mirror(src, dst, label):
    """Copy src -> dst if src has content. Uses rsync, falls back to cp -ru."""
    if not os.path.isdir(src) or not os.listdir(src):
        print(f"  {label}: nothing to restore")
        return
    os.makedirs(dst, exist_ok=True)
    r = subprocess.run(["rsync", "-a", src + "/", dst + "/"], capture_output=True)
    if r.returncode != 0:
        subprocess.run(f'cp -ru "{src}/." "{dst}/"', shell=True, check=False)
    print(f"  {label}: restored -> {dst}")

SIZE = f"size_{BOARD_SIZE:02d}"
PAIRS = [
    (f"{DRIVE_DIR}/models/{SIZE}/{RUN_ID}",  f"{WORK_DIR}/models/{SIZE}/{RUN_ID}",  "models"),
    (f"{DRIVE_DIR}/results/{SIZE}/{RUN_ID}", f"{WORK_DIR}/results/{SIZE}/{RUN_ID}", "results"),
]

print(f"Restoring {RUN_ID} from Drive:")
for src, dst, label in PAIRS:
    _mirror(src, dst, label)

reg = f"{WORK_DIR}/models/{SIZE}/{RUN_ID}/registry.json"
if os.path.isfile(reg):
    import json
    snaps = json.load(open(reg))
    print(f"\n{len(snaps)} existing snapshot(s); latest = "
          f"{snaps[-1]['version_id']} @ {snaps[-1]['games_trained']} games")
else:
    print(f"\nNo existing snapshots — this will be a fresh {RUN_ID}.")

## 6. Device probe (optional but recommended)The network here is small: a 4-block ResNet with 64 channels on an 8×8 board.Training alternates between many *single-state* forward passes (game simulation,evaluation, benchmarking) and a few *batched* forward/backward passes (thegradient steps). GPUs win decisively on the second and can actually lose on thefirst, because per-kernel launch latency dominates a forward pass this small.So whether CUDA is a win here is an empirical question, not a given. This cellmeasures both and tells you which to use. It takes about a minute.

In [ ]:
import subprocess, sys, os, textwrap, json

PROBE = textwrap.dedent("""
    import os, sys, time
    sys.path.insert(0, os.environ["WORK_DIR"])
    import numpy as np, torch
    from src.agents.dqn_agent import DQNAgent
    from src.agents.heuristic_agent import HeuristicAgent
    from src.game.env import GameEnv
    from src.training.replay_buffer import ReplayBuffer
    from src.training.self_play import play_episode
    from src.config import (STATE_CHANNELS_V2, NETWORK_ARCH, BATCH_SIZE,
                            REPLAY_CAPACITY, GRADIENT_STEPS_PER_GAME,
                            USE_SYMMETRY_AUGMENTATION)

    np.random.seed(0); torch.manual_seed(0)
    size, mode = int(os.environ["BS"]), os.environ["MODE"]
    agent  = DQNAgent(size, in_channels=STATE_CHANNELS_V2, network_arch=NETWORK_ARCH)
    agent.set_epsilon(0.1)
    buf    = ReplayBuffer(REPLAY_CAPACITY, use_augmentation=USE_SYMMETRY_AUGMENTATION)
    env    = GameEnv(board_size=size, mode=mode)
    opp    = HeuristicAgent()

    N = 40
    t0 = time.monotonic()
    for i in range(N):
        t1, t2, _ = play_episode(agent, opp, env)
        for t in t1:
            buf.push(t.state, t.action, t.reward, t.next_state, t.done,
                     t.legal_mask_next, weight=1.0, gamma_n=t.gamma_n)
        if len(buf) >= BATCH_SIZE:
            for _ in range(GRADIENT_STEPS_PER_GAME):
                agent.update(buf.sample(BATCH_SIZE))
    dt = time.monotonic() - t0
    print(f"RESULT {agent._device.type} {dt:.2f} {N / dt * 3600:.0f}")
""")

probe_path = "/content/_probe.py"
open(probe_path, "w").write(PROBE)

results = {}
for dev in ("cpu", "cuda"):
    env = {**os.environ, "FLAT4_DEVICE": dev, "WORK_DIR": WORK_DIR,
           "BS": str(BOARD_SIZE), "MODE": MODE}
    r = subprocess.run([sys.executable, probe_path], capture_output=True, text=True,
                       cwd=WORK_DIR, env=env)
    line = [l for l in r.stdout.splitlines() if l.startswith("RESULT")]
    if not line:
        print(f"{dev}: probe failed\n{r.stderr[-800:]}")
        continue
    _, actual, secs, gph = line[0].split()
    results[actual] = float(gph)
    print(f"{dev:>4} -> ran on {actual:<4} {float(secs):6.2f}s  ~{float(gph):,.0f} games/hour")

if len(results) == 2:
    best = max(results, key=results.get)
    margin = results[best] / min(results.values())
    print(f"\nFaster: {best.upper()} by {margin:.2f}x")
    print("(Local laptop baseline for run_ftf_004 was ~150-180 games/hour.)")
    DEVICE = best
else:
    DEVICE = "cuda" if "cuda" in results else "cpu"

print(f"\nDEVICE = {DEVICE!r}  (cell 8 will use this)")

## 7. Build the training commandDetects whether this is a fresh start or a resume and constructs the rightflags. Nothing is executed here — inspect the command, then run cell 8.

In [ ]:
import json, os, shlex, sys

SIZE = f"size_{BOARD_SIZE:02d}"
reg_path = f"{WORK_DIR}/models/{SIZE}/{RUN_ID}/registry.json"

cmd = [sys.executable, "-u", "-m", "src.training.train",
       "--size", str(BOARD_SIZE),
       "--mode", MODE,
       "--games", str(TOTAL_GAMES),
       "--run-id", RUN_ID,
       "--benchmark", BENCHMARK,
       "--benchmark-every", str(BENCHMARK_EVERY),
       "--lr-start", str(LR_START),
       "--seed", str(SEED),
       "--workers", str(WORKERS)]

if os.path.isfile(reg_path):
    snaps = json.load(open(reg_path))
    latest = snaps[-1]
    # Carry Elo across sessions — without this the curve restarts at 800 each time.
    elo = latest.get("elo_rating") or SEED_ELO
    cmd += ["--resume", "--elo-start", str(elo)]
    print(f"RESUMING from {latest['version_id']} @ {latest['games_trained']} games, Elo {elo}")
else:
    cmd += ["--load-weights", SEED_WEIGHTS,
            "--start-game", str(SEED_GAME),
            "--elo-start", str(SEED_ELO)]
    print(f"FRESH START seeded from {SEED_WEIGHTS} at game {SEED_GAME}")

TRAIN_CMD = " ".join(shlex.quote(c) for c in cmd)
print("\n" + TRAIN_CMD)

## 8. Launch trainingStarts the trainer detached (`setsid` + `nohup`) so it survives this cellfinishing, plus a background loop that mirrors progress to Drive every`SYNC_SECONDS`. Both keep running while you do other things.Re-running this cell refuses to start a second trainer if one is already alive.

In [ ]:
import subprocess, os, time, shlex

LOCAL_LOG = "/content/train.log"
SIZE      = f"size_{BOARD_SIZE:02d}"

def trainer_pids():
    r = subprocess.run(["pgrep", "-f", f"src.training.train.*{RUN_ID}"],
                       capture_output=True, text=True)
    return [p for p in r.stdout.split() if p]

try:
    DEVICE
except NameError:
    # Cell 6 (the probe) is optional — fall back to whatever torch reports.
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Probe not run; defaulting to DEVICE={DEVICE!r}")

alive = trainer_pids()
if alive:
    print(f"Trainer already running (pid {', '.join(alive)}). Not starting another.")
    print("Use cell 9 to monitor, or cell 11 to stop it.")
else:
    env = {**os.environ, "FLAT4_DEVICE": DEVICE, "PYTHONUNBUFFERED": "1",
           "PYTHONPATH": WORK_DIR}

    subprocess.Popen(
        f"setsid nohup {TRAIN_CMD} > {shlex.quote(LOCAL_LOG)} 2>&1 < /dev/null &",
        shell=True, cwd=WORK_DIR, env=env,
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )

    # Background mirror: local run dirs + log -> Drive, every SYNC_SECONDS.
    #
    # Deliberately no --delete, and each copy is guarded on a non-empty source.
    # A mirror that can delete would destroy the Drive backup if it ever ran
    # against an empty local dir (e.g. cell 8 run in a fresh session before the
    # restore in cell 5). Copy-only makes that failure mode impossible; a stale
    # leftover file in Drive is harmless by comparison.
    sync = f"""
    mirror () {{
      if [ -d "$1" ] && [ -n "$(ls -A "$1" 2>/dev/null)" ]; then
        mkdir -p "$2"; rsync -a "$1/" "$2/" 2>/dev/null || cp -ru "$1/." "$2/" 2>/dev/null
      fi
    }}
    while true; do
      mirror "{WORK_DIR}/models/{SIZE}/{RUN_ID}"  "{DRIVE_DIR}/models/{SIZE}/{RUN_ID}"
      mirror "{WORK_DIR}/results/{SIZE}/{RUN_ID}" "{DRIVE_DIR}/results/{SIZE}/{RUN_ID}"
      cp -f "{LOCAL_LOG}" "{DRIVE_DIR}/logs/{RUN_ID}.log" 2>/dev/null
      sleep {SYNC_SECONDS}
    done
    """
    open("/content/_sync.sh", "w").write(sync)
    subprocess.Popen("setsid nohup bash /content/_sync.sh > /dev/null 2>&1 < /dev/null &",
                     shell=True)

    time.sleep(20)
    pids = trainer_pids()
    print(f"Trainer pid: {', '.join(pids) if pids else 'NOT RUNNING - check log below'}")
    print(f"Device: {DEVICE}   Drive mirror: every {SYNC_SECONDS}s\n")
    print("--- first 30 lines ---")
    print(subprocess.run(["head", "-30", LOCAL_LOG], capture_output=True, text=True).stdout)

## 9. MonitorSafe to run repeatedly, and safe to run in a fresh session after a reconnect.

In [ ]:
import subprocess, os, csv

LOCAL_LOG = "/content/train.log"
SIZE      = f"size_{BOARD_SIZE:02d}"

pids = subprocess.run(["pgrep", "-f", f"src.training.train.*{RUN_ID}"],
                      capture_output=True, text=True).stdout.split()
print(f"Trainer: {'RUNNING pid ' + pids[0] if pids else 'not running'}")

log_csv = f"{WORK_DIR}/results/{SIZE}/{RUN_ID}/training_log.csv"
if os.path.isfile(log_csv):
    with open(log_csv) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = rows[-1]
        print(f"\ngame {last['game']}/{TOTAL_GAMES}   "
              f"{float(last['games_per_hour']):.0f} games/hour")
        print(f"  mean_ep_len {last['mean_ep_len']}  (lower is better; "
              f"run_ftf_004 best was 8.7)")
        print(f"  WR vs random    {float(last['win_rate_vs_random']):.1%}")
        print(f"  WR vs heuristic {float(last['win_rate_vs_heuristic']):.1%}"
              f"   (run_ftf_004 peaked ~21%)")
        print(f"  Elo {last['elo_rating']}   loss {last['loss']}")

        best = min(rows, key=lambda r: float(r["mean_ep_len"] or 1e9))
        print(f"\nBest mean_ep_len so far: {best['mean_ep_len']} at game {best['game']}")

        remaining = TOTAL_GAMES - int(last["game"])
        gph = float(last["games_per_hour"]) or 1
        print(f"~{remaining / gph:.1f} hours of training left at current rate")

print("\n--- last 25 log lines ---")
print(subprocess.run(["tail", "-25", LOCAL_LOG], capture_output=True, text=True).stdout)

## 10. Push results back to GitHubOptional — the Drive mirror is already your safety net. Use this when a sessionproduces something worth committing.Needs a GitHub personal access token with `repo` scope(github.com → Settings → Developer settings → Personal access tokens).`getpass` keeps it out of the saved notebook, but it is still held in memory forthe session, so use a short-lived token and revoke it when the project is done.

In [ ]:
import subprocess, getpass, os

SIZE = f"size_{BOARD_SIZE:02d}"

subprocess.run(["git", "config", "user.name", "colile1"], cwd=WORK_DIR, check=True)
subprocess.run(["git", "config", "user.email", "colilesibanda@gmail.com"], cwd=WORK_DIR, check=True)

paths = [f"models/{SIZE}/{RUN_ID}", f"results/{SIZE}/{RUN_ID}"]
subprocess.run(["git", "add", "--", *paths], cwd=WORK_DIR, check=True)

status = subprocess.run(["git", "status", "--short", "--", *paths],
                        cwd=WORK_DIR, capture_output=True, text=True).stdout
if not status.strip():
    print("Nothing to commit.")
else:
    print("Staged:\n" + status)
    n = subprocess.run(["git", "diff", "--cached", "--numstat", "--", *paths],
                       cwd=WORK_DIR, capture_output=True, text=True).stdout
    print(f"({len(n.strip().splitlines())} files)\n")

    games = "?"
    reg = f"{WORK_DIR}/models/{SIZE}/{RUN_ID}/registry.json"
    if os.path.isfile(reg):
        import json
        games = json.load(open(reg))[-1]["games_trained"]

    msg = f"{RUN_ID}: Colab GPU training through game {games}"
    subprocess.run(["git", "commit", "-m", msg], cwd=WORK_DIR, check=True)

    token = getpass.getpass("GitHub token (input hidden): ").strip()
    url = REPO_URL.replace("https://", f"https://{token}@")
    push = subprocess.run(["git", "push", url, f"HEAD:{BRANCH}"],
                          cwd=WORK_DIR, capture_output=True, text=True)
    # Never print push output verbatim — the remote URL embeds the token.
    print("Push OK" if push.returncode == 0 else "Push FAILED (check token/permissions)")
    del token, url

## 11. Stop training / download a bundleTwo independent utilities — run whichever you need.

In [ ]:
# --- Stop the trainer and the Drive mirror ---
import subprocess
subprocess.run(["pkill", "-f", f"src.training.train.*{RUN_ID}"], check=False)
subprocess.run(["pkill", "-f", "_sync.sh"], check=False)
print("Stopped. Progress up to the last mirror is safe in Drive.")

In [ ]:
# --- Force an immediate mirror to Drive, then build a downloadable zip ---
import subprocess, os

SIZE = f"size_{BOARD_SIZE:02d}"
for kind in ("models", "results"):
    src, dst = f"{WORK_DIR}/{kind}/{SIZE}/{RUN_ID}", f"{DRIVE_DIR}/{kind}/{SIZE}/{RUN_ID}"
    if os.path.isdir(src) and os.listdir(src):
        os.makedirs(dst, exist_ok=True)
        subprocess.run(["rsync", "-a", src + "/", dst + "/"], check=False)
print("Mirrored to Drive.")

bundle = f"/content/{RUN_ID}.zip"
subprocess.run(["zip", "-qr", bundle,
                f"models/{SIZE}/{RUN_ID}", f"results/{SIZE}/{RUN_ID}"],
               cwd=WORK_DIR, check=False)
print(f"{bundle}  ({os.path.getsize(bundle) / 2**20:.1f} MiB)")

# Also drop a copy in Drive — files.download() only works in the browser
# Colab UI, not in a VS Code session attached to a Colab runtime.
subprocess.run(["cp", "-f", bundle, f"{DRIVE_DIR}/{RUN_ID}.zip"], check=False)
print(f"Copied to {DRIVE_DIR}/{RUN_ID}.zip")

try:
    from google.colab import files
    files.download(bundle)
except Exception as e:
    print(f"Direct download unavailable ({type(e).__name__}) — use the Drive copy.")